# Data Exploration Notebook

This notebook consolidates the exploration scripts into a single, parameterized workflow.
Set the paths in the **Configuration** section, then run the sections you need.
Most sections save figures to disk and also display them inline.

In [ ]:
# Core imports
from __future__ import annotations

import csv
import gc
import json
import math
import os
import re
import glob
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Iterator

import cv2
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
import xml.etree.ElementTree as ET

import openslide

from src.data.ndpa_reader import NDPAData
from src.data.ndpi_reader import NDPIData
from src.preprocessing.focus_metrics import (
    tenengrad,
    variance_of_laplacian,
    percentile_vol,
    percentile_tenengrad,
)
from src.preprocessing.h5_utils import list_h5_paths, list_tile_jobs

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

COLOR_VOL = "#0077BB"
COLOR_TEN = "#EE3377"
COLOR_PVOL = "#009944"
COLOR_PTEN = "#FF8C00"

## Configuration

In [ ]:
# Base paths
BASE_DIR = Path(".").resolve()
EXPLORATION_DIR = BASE_DIR / "src" / "exploration"
OUTPUT_DIR = BASE_DIR / "output" / "exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input paths (edit as needed)
ANNOTATION_CLASS_CSV = EXPLORATION_DIR / "annotation_by_category.csv"
ANNOTATION_COUNTS_TXT = EXPLORATION_DIR / "annotation_counts.txt"
NDPA_DIR = BASE_DIR / "output" / "annotator"  # or wherever NDPA files live
NDPI_DIR = BASE_DIR / "output" / "annotator"  # or wherever NDPI files live
STAIN_CSV = EXPLORATION_DIR / "Samples_with_annotations_stained_non_stained.csv"
SPLIT_JSON = EXPLORATION_DIR / "train_val_test.json"

# H5 tiles for focus exploration
H5_INPUT = BASE_DIR / "output" / "tiles"  # directory of .h5 files

# Visualization constants
CLASSES = ["pol", "spo", "paly", "din", "alg", "fun"]

print("Output dir:", OUTPUT_DIR)
print("Annotation CSV exists:", ANNOTATION_CLASS_CSV.exists())
print("Annotation counts exists:", ANNOTATION_COUNTS_TXT.exists())
print("Stain CSV exists:", STAIN_CSV.exists())
print("Split JSON exists:", SPLIT_JSON.exists())

## Annotation Class Distribution

In [ ]:
if ANNOTATION_CLASS_CSV.exists():
    df = pd.read_csv(ANNOTATION_CLASS_CSV)
    df = df.set_index(df.columns[0])
    annotation_df = df.loc[CLASSES]
    annotation_df = annotation_df.apply(pd.to_numeric, errors="coerce")

    combined_freq = annotation_df.sum(axis=1).sort_values(ascending=False)
    total = combined_freq.sum()
    percentages = (combined_freq / total) * 100

    plt.figure(figsize=(9, 5))
    ax = combined_freq.plot(kind="bar")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.xlabel("Annotation Class", fontsize=12)
    plt.ylabel("Combined Frequency", fontsize=12)
    plt.title("Distribution of Annotation Classes", fontsize=14)
    plt.xticks(rotation=0)

    for i, (count, pct) in enumerate(zip(combined_freq, percentages)):
        ax.text(i, count, f"{int(count)}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)

    out_path = OUTPUT_DIR / "annotation_class_distribution.png"
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    print("Saved", out_path)
else:
    print("Missing annotation_by_category.csv. Update ANNOTATION_CLASS_CSV.")

## Annotation Count Histogram

In [ ]:
if ANNOTATION_COUNTS_TXT.exists():
    with open(ANNOTATION_COUNTS_TXT, "r", encoding="utf-8") as file:
        data = [int(line.split()[-1]) for line in file if line.strip()]

    plt.hist(data, bins="auto")
    plt.xlabel("Number of Annotated Palynomorphs")
    plt.ylabel("Number of Images")
    plt.title("Distribution of Annotations per Image")
    out_path = OUTPUT_DIR / "annotation_count_histogram.png"
    plt.savefig(out_path)
    plt.show()
    print("Saved", out_path)
else:
    print("Missing annotation_counts.txt. Update ANNOTATION_COUNTS_TXT.")

## Annotation Size Distribution (Circle Annotations)

In [ ]:
MPP_X = 0.22885913720105275
MPP_Y = 0.22886437497139195
NM_PER_PX_X = MPP_X * 1000.0
NM_PER_PX_Y = MPP_Y * 1000.0
MIN_DIAMETER_UM = 1
MAX_DIAMETER_UM = 250

files = sorted(NDPA_DIR.glob("*.ndpi.ndpa"))
if not files:
    print("No NDPA files found in", NDPA_DIR)
else:
    rows = []
    for path in files:
        root = ET.parse(path).getroot()
        for ann in root.iter():
            if ann.tag.split("}")[-1] != "annotation":
                continue
            if ann.attrib.get("displayname") != "AnnotateCircle":
                continue
            if ann.attrib.get("type") != "circle":
                continue

            radius_nm = None
            for child in list(ann):
                if child.tag.split("}")[-1] == "radius":
                    txt = (child.text or "").strip()
                    try:
                        radius_nm = float(txt)
                    except ValueError:
                        radius_nm = None
                    break
            if radius_nm is None:
                continue

            diameter_nm = 2.0 * radius_nm
            diameter_um = diameter_nm / 1000.0
            if not (MIN_DIAMETER_UM <= diameter_um <= MAX_DIAMETER_UM):
                continue

            diameter_px_x = diameter_nm / NM_PER_PX_X
            diameter_px_y = diameter_nm / NM_PER_PX_Y
            diameter_px = (diameter_px_x + diameter_px_y) / 2.0

            rows.append({
                "ndpa_file": str(path),
                "radius_nm": radius_nm,
                "diameter_um": diameter_um,
                "diameter_px": diameter_px,
            })

    if not rows:
        print("No circle annotations extracted after filtering.")
    else:
        df = pd.DataFrame(rows)
        df.to_csv(OUTPUT_DIR / "circle_sizes_summary.csv", index=False)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(df["diameter_px"], bins=30, edgecolor="black")
        axes[0].set_title("Small Annotation Diameter (Pixels)")
        axes[0].set_xlabel("Diameter (pixels)")
        axes[0].set_ylabel("Count")

        axes[1].hist(df["diameter_um"], bins=30, edgecolor="black")
        axes[1].set_title("Small Annotation Diameter (Microns)")
        axes[1].set_xlabel("Diameter (microns)")
        axes[1].set_ylabel("Count")

        for ax in axes:
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.ticklabel_format(style="plain", useOffset=False)
            ax.xaxis.set_major_locator(plt.MaxNLocator(6))

        fig.suptitle(f"Distribution of Small Annotation Diameters (N={len(df)})", fontsize=14)
        plt.tight_layout()
        out_path = OUTPUT_DIR / "small_annotation_diameter_2plots.png"
        plt.savefig(out_path, dpi=300)
        plt.show()
        print("Saved", out_path)

## Annotation Density (Per NDPI/NDPA Pair)

In [ ]:
TILE_SIZE_PX = 1024

density_csv = OUTPUT_DIR / "annotation_density.csv"
if not NDPI_DIR.exists():
    print("NDPI_DIR not found. Update NDPI_DIR to run density.")
else:
    folder = NDPI_DIR
    all_files = {p.name for p in folder.iterdir() if p.is_file()}
    ndpi_files = {f for f in all_files if f.endswith(".ndpi")}
    ndpa_files = {f for f in all_files if f.endswith(".ndpi.ndpa")}

    pairs = []
    orphans = []
    for ndpi_name in ndpi_files:
        expected_ndpa = ndpi_name + ".ndpa"
        if expected_ndpa in ndpa_files:
            pairs.append((ndpi_name, expected_ndpa))
        else:
            orphans.append(ndpi_name)
    for ndpa_name in ndpa_files:
        expected_ndpi = ndpa_name[:-5]
        if expected_ndpi not in ndpi_files:
            orphans.append(ndpa_name)

    if not pairs:
        print("No valid NDPI/NDPA pairs found in", folder)
    else:
        fieldnames = ["file_name", "n_annotations", "n_rois", "n_tiles", "density"]
        errors = []
        with open(density_csv, "w", newline="") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for ndpi_name, ndpa_name in pairs:
                ndpi_path = folder / ndpi_name
                ndpa_path = folder / ndpa_name
                try:
                    ndpa = NDPAData(str(ndpa_path))
                    ndpi = NDPIData(str(ndpi_path))
                    if not ndpa.rois:
                        raise ValueError("No ROI rectangles found in the NDPA file.")

                    n_annotations = len(ndpa.palynomorphs)
                    n_tiles = 0
                    for roi in ndpa.rois:
                        nm_x, nm_y, nm_w, nm_h = roi.bounds.get_bounding_box()
                        meta = ndpi.metadata
                        x0_tmp = (nm_x - meta.x_offset_nm) * 0.001 / meta.mpp_x + meta.full_width / 2
                        y0_tmp = (nm_y - meta.y_offset_nm) * 0.001 / meta.mpp_y + meta.full_height / 2
                        x1_tmp = (nm_x + nm_w - meta.x_offset_nm) * 0.001 / meta.mpp_x + meta.full_width / 2
                        y1_tmp = (nm_y + nm_h - meta.y_offset_nm) * 0.001 / meta.mpp_y + meta.full_height / 2
                        x0, x1 = min(x0_tmp, x1_tmp), max(x0_tmp, x1_tmp)
                        y0, y1 = min(y0_tmp, y1_tmp), max(y0_tmp, y1_tmp)
                        area_w = x1 - x0
                        area_h = y1 - y0
                        tiles_x = math.ceil(area_w / TILE_SIZE_PX)
                        tiles_y = math.ceil(area_h / TILE_SIZE_PX)
                        n_tiles += (tiles_x * tiles_y)

                    density = n_annotations / n_tiles if n_tiles > 0 else float("nan")
                    writer.writerow({
                        "file_name": ndpi_name,
                        "n_annotations": n_annotations,
                        "n_rois": len(ndpa.rois),
                        "n_tiles": n_tiles,
                        "density": f"{density:.6f}",
                    })
                except Exception as exc:
                    errors.append((ndpi_name, str(exc)))

        print("Wrote", density_csv)
        if orphans:
            print("Orphaned files:")
            for name in orphans:
                print("-", name)
        if errors:
            print("Errors:")
            for name, reason in errors:
                print(f"- {name}: {reason}")

## Label Counts and Missing Labels

In [ ]:
# Example usage
# label_counter = Counter()
# for ndpa_path in Path(NDPA_DIR).rglob("*.ndpa"):
#     try:
#         ndpa = NDPAData(str(ndpa_path))
#         for ann in getattr(ndpa, "palynomorphs", []):
#             label = getattr(ann, "label", None)
#             if label:
#                 label_counter[label] += 1
#     except Exception as exc:
#         print(f"Error reading {ndpa_path}: {exc}")
#
# annotation_map = pd.read_csv(EXPLORATION_DIR / "annotation_map.csv")
# csv_labels = set(annotation_map["Specimen_name"].dropna().astype(str).str.lower())
# print("NDPA labels not found in CSV (case-insensitive):")
# for ndpa_label in sorted(label_counter.keys()):
#     if ndpa_label.lower() not in csv_labels:
#         print(ndpa_label)

## Image Staining Distribution

In [ ]:
if STAIN_CSV.exists():
    files = []
    stained = []
    with open(STAIN_CSV, "r", encoding="utf-8") as data:
        reader = csv.reader(data)
        next(reader)
        for row in reader:
            files.append(row[0])
            stained.append(row[1])

    counts = [0, 0]
    for s in stained:
        if s == "stained":
            counts[0] += 1
        else:
            counts[1] += 1

    fig, ax = plt.subplots()
    stained_labels = ["stained", "not stained"]
    bar_colors = ["tab:red", "tab:blue"]
    ax.bar(stained_labels, counts, color=bar_colors)
    ax.set_ylabel("occurrences")
    ax.set_title("Distribution of Image Staining")

    out_path = OUTPUT_DIR / "image_metadata_bar_plot.png"
    plt.savefig(out_path, format="png")
    plt.show()
    print("Saved", out_path)
else:
    print("Missing stain CSV. Update STAIN_CSV.")

## Image Size Distributions (Slides and ROI)

In [ ]:
if NDPI_DIR.exists():
    pixel_width = []
    pixel_height = []
    micron_width = []
    micron_height = []

    for i, entry in enumerate(NDPI_DIR.iterdir(), start=1):
        if entry.is_dir() or entry.suffix != ".ndpi":
            continue
        try:
            with openslide.OpenSlide(str(entry)) as slide:
                props = slide.properties
                w, h = slide.dimensions
                mpp_x = float(props.get("openslide.mpp-x", 0))
                mpp_y = float(props.get("openslide.mpp-y", 0))
        except Exception as exc:
            print(f"Error reading {entry}: {exc}")
            continue

        pixel_width.append(w)
        pixel_height.append(h)
        micron_width.append(mpp_x * w)
        micron_height.append(mpp_y * h)
        if i % 10 == 0:
            print(f"Gathered data from {i} NDPI files")
            gc.collect()

    df = pd.DataFrame({
        "Pixels": np.concatenate([np.array(pixel_width), np.array(pixel_height)]),
        "Dimension": ["Width"] * len(pixel_width) + ["Height"] * len(pixel_height),
    })
    g = sns.FacetGrid(df, row="Dimension", hue="Dimension", aspect=4, height=3, palette=["red", "blue"])
    g.map(sns.histplot, "Pixels", bins=8)
    g.figure.suptitle("Pixel Dimension Histogram")
    g.figure.tight_layout()
    g.figure.subplots_adjust(top=0.9)
    out_path = OUTPUT_DIR / "slide_pixel_dimensions_histogram.png"
    plt.savefig(out_path, format="png")
    plt.show()
    print("Saved", out_path)

    df = pd.DataFrame({
        "Microns": np.concatenate([np.array(micron_width), np.array(micron_height)]),
        "Dimension": ["Width"] * len(micron_width) + ["Height"] * len(micron_height),
    })
    g = sns.FacetGrid(df, row="Dimension", hue="Dimension", aspect=4, height=3, palette=["red", "blue"])
    g.map(sns.histplot, "Microns", bins=8)
    g.figure.suptitle("Micron Dimensions Histogram")
    g.figure.tight_layout()
    g.figure.subplots_adjust(top=0.9)
    out_path = OUTPUT_DIR / "slide_micron_dimensions_histogram.png"
    plt.savefig(out_path, format="png")
    plt.show()
    print("Saved", out_path)
else:
    print("NDPI_DIR not found. Update NDPI_DIR.")

if NDPA_DIR.exists():
    roi_pixel_width = []
    roi_pixel_height = []
    roi_microns_width = []
    roi_microns_height = []

    for i, entry in enumerate(NDPA_DIR.iterdir(), start=1):
        if entry.is_dir() or not entry.name.endswith(".ndpa"):
            continue
        split_file_path = entry.name.split(".")
        ndpi_file_path = entry.parent / f"{split_file_path[0]}.{split_file_path[1]}"

        try:
            with openslide.OpenSlide(str(ndpi_file_path)) as slide:
                props = slide.properties
                mpp_x = float(props.get("openslide.mpp-x", 0))
                mpp_y = float(props.get("openslide.mpp-y", 0))
        except Exception as exc:
            print(f"Error reading {ndpi_file_path}: {exc}")
            continue

        try:
            annotations = NDPAData(str(entry))
            bounding_box_dimensions = [r.bounds.get_bounding_box()[2:] for r in annotations.rois]
        except Exception as exc:
            print(f"Error parsing NDPA file {entry}: {exc}")
            continue

        if not bounding_box_dimensions:
            continue
        for dimensions in bounding_box_dimensions:
            microns_w, microns_h = dimensions[0] / 1000, dimensions[1] / 1000
            pixels_w, pixels_h = microns_w / mpp_x, microns_h / mpp_y
            roi_pixel_width.append(pixels_w)
            roi_pixel_height.append(pixels_h)
            roi_microns_width.append(microns_w)
            roi_microns_height.append(microns_h)
        if i % 10 == 0:
            print(f"Gathered data from {i} NDPA files")
            gc.collect()

    df = pd.DataFrame({
        "Pixels": np.concatenate([np.array(roi_pixel_width), np.array(roi_pixel_height)]),
        "Dimension": ["Width"] * len(roi_pixel_width) + ["Height"] * len(roi_pixel_height),
    })
    g = sns.FacetGrid(df, row="Dimension", hue="Dimension", aspect=4, height=3, palette=["red", "blue"])
    g.map(sns.histplot, "Pixels", bins=8)
    g.figure.suptitle("ROI Pixel Dimensions Histogram")
    g.figure.tight_layout()
    g.figure.subplots_adjust(top=0.9)
    out_path = OUTPUT_DIR / "roi_pixel_dimensions_histogram.png"
    plt.savefig(out_path, format="png")
    plt.show()
    print("Saved", out_path)

    df = pd.DataFrame({
        "Microns": np.concatenate([np.array(roi_microns_width), np.array(roi_microns_height)]),
        "Dimension": ["Width"] * len(roi_microns_width) + ["Height"] * len(roi_microns_height),
    })
    g = sns.FacetGrid(df, row="Dimension", hue="Dimension", aspect=4, height=3, palette=["red", "blue"])
    g.map(sns.histplot, "Microns", bins=8)
    g.figure.suptitle("ROI Micron Dimensions Histogram")
    g.figure.tight_layout()
    g.figure.subplots_adjust(top=0.9)
    out_path = OUTPUT_DIR / "roi_microns_dimensions_histogram.png"
    plt.savefig(out_path, format="png")
    plt.show()
    print("Saved", out_path)
else:
    print("NDPA_DIR not found. Update NDPA_DIR.")

## Split Sanity Check (Train / Val / Test)

In [ ]:
NM_PER_PX = ((MPP_X + MPP_Y) / 2) * 1000
SPLITS = ["train", "val", "test"]

def normalize_name(text: str) -> str:
    text = str(text).lower().replace(".ndpi.ndpa", "").replace(".ndpa", "").replace(".ndpi", "")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

def score_match(a: str, b: str) -> tuple[int, int, int]:
    a, b = normalize_name(a), normalize_name(b)
    if a == b:
        return (4, 9999, 0)
    if a.startswith(b) or b.startswith(a):
        return (3, min(len(a), len(b)), -abs(len(a) - len(b)))
    if a in b or b in a:
        return (2, min(len(a), len(b)), -abs(len(a) - len(b)))
    ta, tb = set(a.split("_")), set(b.split("_"))
    return (1, len(ta & tb), -abs(len(ta) - len(tb)))

def best_match(name: str, candidates: list[str], used: set[str] | None = None, min_overlap: int = 2) -> str | None:
    pool = [c for c in candidates if used is None or c not in used] or candidates
    if not pool:
        return None
    score, match = max(((score_match(name, c), c) for c in pool), key=lambda x: x[0])
    return match if score[0] >= 2 or (score[0] == 1 and score[1] >= min_overlap) else None

def parse_circle_annotations_for_split(ndpa_path: Path) -> list[dict]:
    rows = []
    root = ET.parse(ndpa_path).getroot()
    for ann in root.iter():
        if ann.tag.split("}")[-1] != "annotation":
            continue
        if ann.attrib.get("displayname") != "AnnotateCircle" or ann.attrib.get("type") != "circle":
            continue
        radius = next((c.text for c in ann if c.tag.split("}")[-1] == "radius"), None)
        try:
            diameter_nm = 2 * float(radius)
        except (TypeError, ValueError):
            continue
        diameter_um = diameter_nm / 1000
        if not (MIN_DIAMETER_UM <= diameter_um <= MAX_DIAMETER_UM):
            continue
        rows.append({
            "diameter_um": diameter_um,
            "diameter_px": diameter_nm / NM_PER_PX,
            "source_file": ndpa_path.name,
        })
    return rows

def style_ax(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.ticklabel_format(style="plain", useOffset=False, axis="y")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

def label_bars(ax, values, total: float | None = None, show_count: bool = False) -> None:
    for i, v in enumerate(values):
        text = f"{int(v)}" if show_count or total in (None, 0) else f"{v / total * 100:.1f}%"
        if total not in (None, 0) and show_count:
            text = f"{int(v)}\n({v / total * 100:.1f}%)"
        ax.text(i, v, text, ha="center", va="bottom", fontsize=9)

def save_bar(series: pd.Series, title: str, ylabel: str, out: Path, total: float | None = None, show_count: bool = False) -> None:
    plt.figure(figsize=(8, 5))
    ax = series.plot(kind="bar")
    ax.set_title(title)
    ax.set_xlabel(series.index.name or "")
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=0)
    style_ax(ax)
    label_bars(ax, series.values, total=total, show_count=show_count)
    plt.tight_layout()
    plt.savefig(out, dpi=300)
    plt.close()

if False:
    for path in [SPLIT_JSON, ANNOTATION_CLASS_CSV, STAIN_CSV]:
        if not path.exists():
            print("Missing required file:", path)
            raise SystemExit(1)

    split = json.loads(SPLIT_JSON.read_text(encoding="utf-8"))
    df_classes_raw = pd.read_csv(ANNOTATION_CLASS_CSV).set_index(pd.read_csv(ANNOTATION_CLASS_CSV).columns[0])
    df_classes = df_classes_raw.loc[CLASSES].apply(pd.to_numeric, errors="coerce")
    class_cols = df_classes.columns.tolist()

    df_stain = pd.read_csv(STAIN_CSV)
    df_stain = df_stain.rename(columns={df_stain.columns[0]: "file_name", df_stain.columns[1]: "stain_status"})
    df_stain["stain_status"] = df_stain["stain_status"].astype(str).str.strip().str.lower()

    ndpa_files = sorted(EXPLORATION_DIR.glob("*.ndpa"))
    ndpa_stems = [p.name.replace(".ndpi.ndpa", "").replace(".ndpa", "") for p in ndpa_files]
    ndpa_map = dict(zip(ndpa_stems, ndpa_files))

    csv_rows, ndpa_rows, stain_rows = [], [], []
    used_csv, used_ndpa, used_stain = set(), set(), set()
    stain_candidates = df_stain["file_name"].tolist()

    for split_name in SPLITS:
        for sample in split.get(split_name, []):
            csv_match = best_match(sample, class_cols, used_csv)
            if csv_match:
                used_csv.add(csv_match)
            csv_rows.append({"split": split_name, "sample_name": sample, "matched_csv_column": csv_match})

            ndpa_match = best_match(sample, ndpa_stems, used_ndpa)
            if ndpa_match:
                used_ndpa.add(ndpa_match)
            ndpa_rows.append({
                "split": split_name,
                "sample_name": sample,
                "matched_ndpa_stem": ndpa_match,
                "matched_ndpa_file": ndpa_map.get(ndpa_match),
            })

            stain_match = best_match(sample, stain_candidates)
            stain_status = None
            if stain_match is not None:
                idxs = df_stain.index[df_stain["file_name"] == stain_match].tolist()
                idx = next((i for i in idxs if i not in used_stain), idxs[0] if idxs else None)
                if idx is not None:
                    used_stain.add(idx)
                    stain_status = df_stain.loc[idx, "stain_status"]
            stain_rows.append({
                "split": split_name,
                "sample_name": sample,
                "matched_stain_name": stain_match,
                "stain_status": stain_status,
            })

    df_csv_matches = pd.DataFrame(csv_rows)
    df_ndpa_matches = pd.DataFrame(ndpa_rows)
    df_stain_matches = pd.DataFrame(stain_rows)

    split_class_counts, split_total_counts = {}, {}
    for split_name in SPLITS:
        matched = df_csv_matches.loc[
            (df_csv_matches["split"] == split_name) & df_csv_matches["matched_csv_column"].notna(),
            "matched_csv_column",
        ].tolist()
        counts = df_classes[matched].sum(axis=1).reindex(CLASSES).fillna(0) if matched else pd.Series(0, index=CLASSES)
        split_class_counts[split_name] = counts
        split_total_counts[split_name] = float(counts.sum())

    size_rows = []
    for _, row in df_ndpa_matches.iterrows():
        if pd.notna(row["matched_ndpa_file"]):
            for x in parse_circle_annotations_for_split(Path(row["matched_ndpa_file"])):
                x.update({"split": row["split"], "sample_name": row["sample_name"]})
                size_rows.append(x)
    df_sizes = pd.DataFrame(size_rows)

    split_stain_counts = {}
    for split_name in SPLITS:
        counts = df_stain_matches.loc[
            (df_stain_matches["split"] == split_name) & df_stain_matches["stain_status"].notna(),
            "stain_status",
        ].value_counts()
        stained = int(counts.get("stained", 0))
        not_stained = int(counts.get("no stained", 0)) + int(counts.get("not stained", 0))
        split_stain_counts[split_name] = {"stained": stained, "not_stained": not_stained}

    summary = []
    for split_name in SPLITS:
        counts = split_class_counts[split_name]
        total = split_total_counts[split_name]
        size_sub = df_sizes[df_sizes["split"] == split_name] if not df_sizes.empty else pd.DataFrame()
        row = {
            "split": split_name,
            "total_annotations_from_csv": total,
            "num_size_annotations_from_ndpa": len(size_sub),
            "median_diameter_um": size_sub["diameter_um"].median() if not size_sub.empty else None,
            "mean_diameter_um": size_sub["diameter_um"].mean() if not size_sub.empty else None,
            "median_diameter_px": size_sub["diameter_px"].median() if not size_sub.empty else None,
            "mean_diameter_px": size_sub["diameter_px"].mean() if not size_sub.empty else None,
            "stained_slides": split_stain_counts[split_name]["stained"],
            "not_stained_slides": split_stain_counts[split_name]["not_stained"],
        }
        for cls in CLASSES:
            row[cls] = counts[cls]
            row[f"{cls}_pct"] = counts[cls] / total * 100 if total else 0
        summary.append(row)
    df_summary = pd.DataFrame(summary)

    df_summary.to_csv(OUTPUT_DIR / "split_sanity_summary.csv", index=False)
    df_csv_matches.to_csv(OUTPUT_DIR / "split_csv_matches.csv", index=False)
    df_ndpa_matches.to_csv(OUTPUT_DIR / "split_ndpa_matches.csv", index=False)
    df_stain_matches.to_csv(OUTPUT_DIR / "split_stain_matches.csv", index=False)
    if not df_sizes.empty:
        df_sizes.to_csv(OUTPUT_DIR / "split_size_annotations.csv", index=False)

    save_bar(
        pd.Series(split_total_counts),
        "Total Annotation Number by Split",
        "Total Annotations",
        OUTPUT_DIR / "annotation_number_by_split.png",
        show_count=True,
    )
    for split_name in SPLITS:
        save_bar(
            split_class_counts[split_name],
            f"Annotation Class Distribution: {split_name.capitalize()}",
            "Count",
            OUTPUT_DIR / f"class_distribution_{split_name}.png",
            total=split_total_counts[split_name],
            show_count=True,
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, split_name in zip(axes, SPLITS):
        counts = split_class_counts[split_name]
        counts.plot(kind="bar", ax=ax)
        ax.set_title(split_name.capitalize())
        ax.set_xlabel("Class")
        ax.set_ylabel("Count")
        ax.set_xticklabels(CLASSES, rotation=0)
        style_ax(ax)
        label_bars(ax, counts.values, total=split_total_counts[split_name])
    fig.suptitle("Annotation Class Distribution Across Train / Validation / Test", fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "class_distribution_train_val_test_panel.png", dpi=300)
    plt.close(fig)

    if not df_sizes.empty:
        for metric, xlabel, out in [
            ("diameter_px", "Diameter (pixels)", "size_distribution_pixels_by_split.png"),
            ("diameter_um", "Diameter (microns)", "size_distribution_microns_by_split.png"),
        ]:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
            for ax, split_name in zip(axes, SPLITS):
                sub = df_sizes[df_sizes["split"] == split_name]
                ax.hist(sub[metric], bins=30, edgecolor="black")
                ax.set_title(split_name.capitalize())
                ax.set_xlabel(xlabel)
                ax.set_ylabel("Count")
                style_ax(ax)
                if len(sub):
                    med = sub[metric].median()
                    ax.axvline(med, linestyle="--", linewidth=1.5)
                    ax.text(med, ax.get_ylim()[1] * 0.9, f"median={med:.1f}", rotation=90, va="top", ha="right", fontsize=9)
            fig.suptitle(f"Size Distribution Across Train / Validation / Test: {xlabel}", fontsize=14)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / out, dpi=300)
            plt.close(fig)

    stain_df = pd.DataFrame(split_stain_counts).T[["stained", "not_stained"]]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    stain_df.plot(kind="bar", stacked=True, ax=axes[0])
    axes[0].set_title("Stained vs Not Stained Slides by Split")
    axes[0].set_xlabel("Split")
    axes[0].set_ylabel("Number of Slides")
    axes[0].set_xticklabels(stain_df.index.tolist(), rotation=0)
    style_ax(axes[0])

    stain_df.div(stain_df.sum(axis=1), axis=0).fillna(0).mul(100).plot(kind="bar", stacked=True, ax=axes[1])
    axes[1].set_title("Stained vs Not Stained Proportion by Split")
    axes[1].set_xlabel("Split")
    axes[1].set_ylabel("Percentage")
    axes[1].set_xticklabels(stain_df.index.tolist(), rotation=0)
    style_ax(axes[1])
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "stain_status_by_split.png", dpi=300)
    plt.close(fig)

    print("Saved split sanity outputs to", OUTPUT_DIR)

## Focus Exploration (H5 Tiles)

In [ ]:
def iter_tiles_from_h5(input_path: Path) -> Iterator[tuple[str, np.ndarray, np.ndarray, np.ndarray]]:
    for h5_path in list_h5_paths(str(input_path)):
        image_stem = os.path.splitext(os.path.basename(h5_path))[0]
        with h5py.File(h5_path, "r") as h5f:
            for key in sorted(h5f.keys()):
                if not key.startswith("tile_"):
                    continue
                group = h5f[key]
                data = np.array(group["data"])
                bboxes = np.array(group["bboxes"])
                labels = np.array(group["labels"])
                if bboxes.size == 0:
                    bboxes = bboxes.reshape(0, 4)
                if labels.size == 0:
                    labels = np.array([], dtype=np.int32)
                tile_path = f"{image_stem}/{key}"
                yield tile_path, data, bboxes, labels


def load_single_tile_data(input_path: Path, tile_path: str) -> np.ndarray | None:
    parts = tile_path.split("/", 1)
    if len(parts) != 2:
        return None
    image_stem, group_name = parts
    h5_path = input_path if str(input_path).endswith(".h5") else input_path / f"{image_stem}.h5"
    if not h5_path.is_file():
        return None
    with h5py.File(h5_path, "r") as h5f:
        if group_name not in h5f:
            return None
        return np.array(h5f[group_name]["data"])


def load_single_tile_data_with_stack(input_path: Path, tile_path: str) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    parts = tile_path.split("/", 1)
    if len(parts) != 2:
        return None, None
    image_stem, group_name = parts
    h5_path = input_path if str(input_path).endswith(".h5") else input_path / f"{image_stem}.h5"
    if not h5_path.is_file():
        return None, None
    with h5py.File(h5_path, "r") as h5f:
        if group_name not in h5f:
            return None, None
        group = h5f[group_name]
        data = np.array(group["data"])
        focus_stacked = np.array(group["focus_stacked"])
        return data, focus_stacked


def _score_grayscale(gray: np.ndarray) -> dict[str, float]:
    return {
        "vol": variance_of_laplacian(gray),
        "tenengrad": tenengrad(gray, 3),
        "pvol": percentile_vol(gray, 75.0),
        "ptenengrad": percentile_tenengrad(gray, 3, 75.0),
    }


def _process_tile_chunk(chunk: list[tuple[str, str, str]], filter_clipped: bool) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    roi_records: list[dict[str, Any]] = []
    ds_records: list[dict[str, Any]] = []
    n_z_max = 0
    by_file: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for h5_path, image_stem, group_name in chunk:
        by_file[h5_path].append((image_stem, group_name))

    for h5_path, group_list in by_file.items():
        with h5py.File(h5_path, "r") as h5f:
            for image_stem, group_name in group_list:
                group = h5f[group_name]
                data = np.array(group["data"])
                bboxes = np.array(group["bboxes"])
                labels = np.array(group["labels"])
                if bboxes.size == 0:
                    bboxes = bboxes.reshape(0, 4)
                if labels.size == 0:
                    labels = np.array([], dtype=np.int32)
                tile_path = f"{image_stem}/{group_name}"

                height, width = data.shape[0], data.shape[1]
                n_z = data.shape[3] if data.ndim == 4 else 1
                n_z_max = max(n_z_max, n_z)

                gray_planes = [
                    cv2.cvtColor(data[:, :, :, z] if data.ndim == 4 else data, cv2.COLOR_RGB2GRAY)
                    for z in range(n_z)
                ]

                for z, gray in enumerate(gray_planes):
                    scores = _score_grayscale(gray)
                    ds_records.append({
                        "tile_path": tile_path,
                        "z_index": z,
                        "vol": scores["vol"],
                        "tenengrad": scores["tenengrad"],
                        "pvol": scores["pvol"],
                        "ptenengrad": scores["ptenengrad"],
                    })

                for ann_idx in range(len(labels)):
                    x, y, w, h = bboxes[ann_idx].tolist()
                    x, y, w, h = int(x), int(y), int(w), int(h)
                    label = int(labels[ann_idx])
                    if filter_clipped:
                        if x <= 0 or y <= 0 or x + w >= width or y + h >= height:
                            continue
                    for z, gray in enumerate(gray_planes):
                        crop = gray[y : y + h, x : x + w]
                        if crop.size == 0:
                            continue
                        scores = _score_grayscale(crop)
                        roi_records.append({
                            "tile_path": tile_path,
                            "ann_idx": ann_idx,
                            "label": label,
                            "z_index": z,
                            "vol": scores["vol"],
                            "tenengrad": scores["tenengrad"],
                            "pvol": scores["pvol"],
                            "ptenengrad": scores["ptenengrad"],
                            "bbox": [x, y, w, h],
                        })
    return roi_records, ds_records, n_z_max


def compute_focus_scores_parallel(input_path: Path, filter_clipped: bool = False, n_workers: int = 1) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    jobs = list_tile_jobs(str(input_path))
    if not jobs:
        return [], [], 0
    n_workers = min(n_workers, len(jobs), os.cpu_count() or 1)
    n_workers = max(1, n_workers)
    chunk_size = (len(jobs) + n_workers - 1) // n_workers
    chunks = [jobs[i : i + chunk_size] for i in range(0, len(jobs), chunk_size)]
    roi_records = []
    ds_records = []
    n_z_max = 0
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_process_tile_chunk, chunk, filter_clipped): chunk for chunk in chunks}
        for future in as_completed(futures):
            roi_part, ds_part, n_z = future.result()
            roi_records.extend(roi_part)
            ds_records.extend(ds_part)
            n_z_max = max(n_z_max, n_z)
    return roi_records, ds_records, n_z_max


def compute_focus_scores(tile_stream: Iterator[tuple[str, np.ndarray, np.ndarray, np.ndarray]], filter_clipped: bool = False) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    roi_records: list[dict[str, Any]] = []
    ds_records: list[dict[str, Any]] = []
    n_z_max = 0
    for tile_path, data, bboxes, labels in tile_stream:
        height, width = data.shape[0], data.shape[1]
        n_z = data.shape[3] if data.ndim == 4 else 1
        n_z_max = max(n_z_max, n_z)
        gray_planes = [
            cv2.cvtColor(data[:, :, :, z] if data.ndim == 4 else data, cv2.COLOR_RGB2GRAY)
            for z in range(n_z)
        ]
        for z, gray in enumerate(gray_planes):
            scores = _score_grayscale(gray)
            ds_records.append({
                "tile_path": tile_path,
                "z_index": z,
                "vol": scores["vol"],
                "tenengrad": scores["tenengrad"],
                "pvol": scores["pvol"],
                "ptenengrad": scores["ptenengrad"],
            })
        for ann_idx in range(len(labels)):
            x, y, w, h = bboxes[ann_idx].tolist()
            x, y, w, h = int(x), int(y), int(w), int(h)
            label = int(labels[ann_idx])
            if filter_clipped:
                if x <= 0 or y <= 0 or x + w >= width or y + h >= height:
                    continue
            for z, gray in enumerate(gray_planes):
                crop = gray[y : y + h, x : x + w]
                if crop.size == 0:
                    continue
                scores = _score_grayscale(crop)
                roi_records.append({
                    "tile_path": tile_path,
                    "ann_idx": ann_idx,
                    "label": label,
                    "z_index": z,
                    "vol": scores["vol"],
                    "tenengrad": scores["tenengrad"],
                    "pvol": scores["pvol"],
                    "ptenengrad": scores["ptenengrad"],
                    "bbox": [x, y, w, h],
                })
    return roi_records, ds_records, n_z_max


def _roi_key(rec: dict[str, Any]) -> tuple[str, int]:
    return (rec["tile_path"], rec["ann_idx"])


def plot_best_z_histogram(roi_records: list[dict[str, Any]], n_z: int, output_dir: Path) -> None:
    roi_by_key: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for rec in roi_records:
        roi_by_key[_roi_key(rec)].append(rec)
    best_vol = [max(recs, key=lambda r: r["vol"])["z_index"] for recs in roi_by_key.values()]
    best_ten = [max(recs, key=lambda r: r["tenengrad"])["z_index"] for recs in roi_by_key.values()]
    best_pvol = [max(recs, key=lambda r: r["pvol"])["z_index"] for recs in roi_by_key.values()]
    best_pten = [max(recs, key=lambda r: r["ptenengrad"])["z_index"] for recs in roi_by_key.values()]
    z_bins = np.arange(-0.5, n_z + 0.5, 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
    axes[0, 0].hist(best_vol, bins=z_bins, color=COLOR_VOL, edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[0, 0].set_title("Variance of Laplacian\nBest Focal Plane per Palynomorph")
    axes[0, 0].set_xlabel("Focal Plane Index (Z)")
    axes[0, 0].set_ylabel("Number of Palynomorphs")
    axes[0, 0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    axes[0, 0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    axes[0, 1].hist(best_ten, bins=z_bins, color=COLOR_TEN, edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[0, 1].set_title("Tenengrad Gradient\nBest Focal Plane per Palynomorph")
    axes[0, 1].set_xlabel("Focal Plane Index (Z)")
    axes[0, 1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    axes[1, 0].hist(best_pvol, bins=z_bins, color=COLOR_PVOL, edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[1, 0].set_title("Percentile VoL (p=75)\nBest Focal Plane per Palynomorph")
    axes[1, 0].set_xlabel("Focal Plane Index (Z)")
    axes[1, 0].set_ylabel("Number of Palynomorphs")
    axes[1, 0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    axes[1, 0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    axes[1, 1].hist(best_pten, bins=z_bins, color=COLOR_PTEN, edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[1, 1].set_title("Percentile Tenengrad (p=75)\nBest Focal Plane per Palynomorph")
    axes[1, 1].set_xlabel("Focal Plane Index (Z)")
    axes[1, 1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    fig.suptitle("Distribution of Best Focal Planes Across Annotated Palynomorphs", fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    path = output_dir / "01_best_focal_plane_histogram.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_mean_focus_per_z_roi(roi_records: list[dict[str, Any]], n_z: int, output_dir: Path) -> None:
    z_vol: dict[int, list[float]] = defaultdict(list)
    z_ten: dict[int, list[float]] = defaultdict(list)
    z_pvol: dict[int, list[float]] = defaultdict(list)
    z_pten: dict[int, list[float]] = defaultdict(list)
    for rec in roi_records:
        z_vol[rec["z_index"]].append(rec["vol"])
        z_ten[rec["z_index"]].append(rec["tenengrad"])
        z_pvol[rec["z_index"]].append(rec["pvol"])
        z_pten[rec["z_index"]].append(rec["ptenengrad"])
    zs = sorted(z_vol.keys())
    vol_mean = [np.mean(z_vol[z]) for z in zs]
    vol_std = [np.std(z_vol[z]) for z in zs]
    ten_mean = [np.mean(z_ten[z]) for z in zs]
    ten_std = [np.std(z_ten[z]) for z in zs]
    pvol_mean = [np.mean(z_pvol[z]) for z in zs]
    pvol_std = [np.std(z_pvol[z]) for z in zs]
    pten_mean = [np.mean(z_pten[z]) for z in zs]
    pten_std = [np.std(z_pten[z]) for z in zs]
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.errorbar(zs, vol_mean, yerr=vol_std, fmt="o-", color=COLOR_VOL, capsize=4, label="Variance of Laplacian", linewidth=1.8, markersize=6)
    ax1.errorbar(zs, pvol_mean, yerr=pvol_std, fmt="^-", color=COLOR_PVOL, capsize=4, label="Percentile VoL (p=75)", linewidth=1.8, markersize=6)
    ax1.set_xlabel("Focal Plane Index (Z)")
    ax1.set_ylabel("VoL Score")
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2 = ax1.twinx()
    ax2.errorbar(zs, ten_mean, yerr=ten_std, fmt="s-", color=COLOR_TEN, capsize=4, label="Tenengrad", linewidth=1.8, markersize=6)
    ax2.errorbar(zs, pten_mean, yerr=pten_std, fmt="D-", color=COLOR_PTEN, capsize=4, label="Percentile Tenengrad (p=75)", linewidth=1.8, markersize=6)
    ax2.set_ylabel("Tenengrad Score")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", framealpha=0.9)
    ax1.set_title("Per-ROI Mean Focus Score Across Focal Planes\n(mean +- 1 std over all annotated palynomorphs)")
    fig.tight_layout()
    path = output_dir / "02_per_roi_focus_vs_z.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_dataset_wide_focus(ds_records: list[dict[str, Any]], n_z: int, output_dir: Path) -> None:
    z_vol: dict[int, list[float]] = defaultdict(list)
    z_ten: dict[int, list[float]] = defaultdict(list)
    z_pvol: dict[int, list[float]] = defaultdict(list)
    z_pten: dict[int, list[float]] = defaultdict(list)
    for rec in ds_records:
        z_vol[rec["z_index"]].append(rec["vol"])
        z_ten[rec["z_index"]].append(rec["tenengrad"])
        z_pvol[rec["z_index"]].append(rec["pvol"])
        z_pten[rec["z_index"]].append(rec["ptenengrad"])
    zs = sorted(z_vol.keys())
    vol_mean = [np.mean(z_vol[z]) for z in zs]
    ten_mean = [np.mean(z_ten[z]) for z in zs]
    pvol_mean = [np.mean(z_pvol[z]) for z in zs]
    pten_mean = [np.mean(z_pten[z]) for z in zs]
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.plot(zs, vol_mean, "o-", color=COLOR_VOL, label="Variance of Laplacian", linewidth=2, markersize=7)
    ax1.plot(zs, pvol_mean, "^-", color=COLOR_PVOL, label="Percentile VoL (p=75)", linewidth=2, markersize=7)
    ax1.set_xlabel("Focal Plane Index (Z)")
    ax1.set_ylabel("VoL Score")
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2 = ax1.twinx()
    ax2.plot(zs, ten_mean, "s-", color=COLOR_TEN, label="Tenengrad", linewidth=2, markersize=7)
    ax2.plot(zs, pten_mean, "D-", color=COLOR_PTEN, label="Percentile Tenengrad (p=75)", linewidth=2, markersize=7)
    ax2.set_ylabel("Tenengrad Score")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", framealpha=0.9)
    ax1.set_title("Dataset-Wide Mean Focus Score per Focal Plane\n(averaged over all tiles)")
    fig.tight_layout()
    path = output_dir / "03_dataset_wide_focus_vs_z.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_metric_agreement_scatter(roi_records: list[dict[str, Any]], output_dir: Path) -> None:
    roi_by_key: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for rec in roi_records:
        roi_by_key[_roi_key(rec)].append(rec)
    vol_best_z = [max(recs, key=lambda r: r["vol"])["z_index"] for recs in roi_by_key.values()]
    ten_best_z = [max(recs, key=lambda r: r["tenengrad"])["z_index"] for recs in roi_by_key.values()]
    fig, ax = plt.subplots(figsize=(6, 6))
    jitter = 0.15
    rng = np.random.default_rng(42)
    jx = rng.uniform(-jitter, jitter, len(vol_best_z))
    jy = rng.uniform(-jitter, jitter, len(ten_best_z))
    ax.scatter(np.array(vol_best_z) + jx, np.array(ten_best_z) + jy, alpha=0.6, s=50, edgecolors="white", linewidth=0.5, color="#6A4C93")
    lims = [min(min(vol_best_z), min(ten_best_z)) - 0.5, max(max(vol_best_z), max(ten_best_z)) + 0.5]
    ax.plot(lims, lims, "--", color="gray", linewidth=1, label="Perfect agreement")
    ax.set_xlabel("Best Focal Plane (Variance of Laplacian)")
    ax.set_ylabel("Best Focal Plane (Tenengrad)")
    ax.set_title("Agreement Between Focus Metrics on Best Focal Plane per Palynomorph")
    ax.legend(loc="upper left", framealpha=0.9)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.set_aspect("equal", adjustable="box")
    agree = sum(1 for v, t in zip(vol_best_z, ten_best_z) if v == t)
    total = len(vol_best_z)
    ax.text(0.95, 0.05, f"Agreement: {agree}/{total} ({100 * agree / max(total, 1):.1f}%)", transform=ax.transAxes, ha="right", va="bottom", fontsize=10, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
    fig.tight_layout()
    path = output_dir / "04_metric_agreement_scatter.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_heatmap(roi_records: list[dict[str, Any]], n_z: int, output_dir: Path) -> None:
    roi_by_key: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for rec in roi_records:
        roi_by_key[_roi_key(rec)].append(rec)
    sorted_keys = sorted(roi_by_key.keys())
    row_labels = []
    vol_matrix = []
    ten_matrix = []
    pvol_matrix = []
    pten_matrix = []
    for key in sorted_keys:
        recs = sorted(roi_by_key[key], key=lambda r: r["z_index"])
        row_labels.append(f"label {recs[0]['label']} ({os.path.basename(key[0])}, #{key[1]})")
        vol_row = [0.0] * n_z
        ten_row = [0.0] * n_z
        pvol_row = [0.0] * n_z
        pten_row = [0.0] * n_z
        for rec in recs:
            vol_row[rec["z_index"]] = rec["vol"]
            ten_row[rec["z_index"]] = rec["tenengrad"]
            pvol_row[rec["z_index"]] = rec["pvol"]
            pten_row[rec["z_index"]] = rec["ptenengrad"]
        vol_matrix.append(vol_row)
        ten_matrix.append(ten_row)
        pvol_matrix.append(pvol_row)
        pten_matrix.append(pten_row)
    vol_matrix = np.array(vol_matrix)
    ten_matrix = np.array(ten_matrix)
    pvol_matrix = np.array(pvol_matrix)
    pten_matrix = np.array(pten_matrix)
    max_rows = 50
    if len(row_labels) > max_rows:
        show_indices = list(range(max_rows // 2)) + list(range(len(row_labels) - max_rows // 2, len(row_labels)))
        row_labels = [row_labels[i] for i in show_indices]
        vol_matrix = vol_matrix[show_indices]
        ten_matrix = ten_matrix[show_indices]
        pvol_matrix = pvol_matrix[show_indices]
        pten_matrix = pten_matrix[show_indices]
    n_rows = len(row_labels)
    fig_height = max(5, 0.35 * n_rows + 2)
    fig, axes = plt.subplots(2, 2, figsize=(18, fig_height))
    for ax, matrix, title, label in [
        (axes[0, 0], vol_matrix, "Variance of Laplacian\nper Palynomorph x Focal Plane", "VoL Score"),
        (axes[0, 1], ten_matrix, "Tenengrad Gradient\nper Palynomorph x Focal Plane", "Tenengrad Score"),
        (axes[1, 0], pvol_matrix, "Percentile VoL (p=75)\nper Palynomorph x Focal Plane", "pVoL Score"),
        (axes[1, 1], pten_matrix, "Percentile Tenengrad (p=75)\nper Palynomorph x Focal Plane", "pTenengrad Score"),
    ]:
        cmap = "viridis" if "VoL" in label else "inferno"
        im = ax.imshow(matrix, aspect="auto", cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("Focal Plane Index (Z)")
        ax.set_ylabel("Annotated Palynomorph")
        ax.set_yticks(range(n_rows))
        ax.set_yticklabels(row_labels, fontsize=7)
        ax.set_xticks(range(n_z))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=label)
    fig.suptitle("Focus Score Heatmap - All Palynomorphs Across Focal Planes", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    path = output_dir / "05_focus_heatmap.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_best_worst_examples(roi_records: list[dict[str, Any]], input_path: Path, output_dir: Path, n_examples: int = 8, metric: str = "vol", mode: str = "spread") -> None:
    import random

    roi_by_key: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for rec in roi_records:
        roi_by_key[_roi_key(rec)].append(rec)
    roi_infos = []
    for key, recs in roi_by_key.items():
        best_rec = max(recs, key=lambda r: r[metric])
        worst_rec = min(recs, key=lambda r: r[metric])
        roi_infos.append({
            "key": key,
            "best_rec": best_rec,
            "worst_rec": worst_rec,
            "spread": best_rec[metric] - worst_rec[metric],
            "label": best_rec["label"],
        })
    if mode == "random":
        random.seed(67)
        selected = random.sample(roi_infos, min(n_examples, len(roi_infos)))
    else:
        roi_infos.sort(key=lambda r: r["spread"], reverse=True)
        selected = roi_infos[:n_examples]
    if not selected:
        return
    n = len(selected)
    fig, axes = plt.subplots(n, 2, figsize=(6, 2.5 * n))
    if n == 1:
        axes = np.atleast_2d(axes)
    metric_label = {"vol": "VoL", "tenengrad": "Tenengrad", "pvol": "pVoL", "ptenengrad": "pTenengrad"}.get(metric, metric)
    for i, info in enumerate(selected):
        tile_path, ann_idx = info["key"]
        best_z = info["best_rec"]["z_index"]
        worst_z = info["worst_rec"]["z_index"]
        bx, by, bw, bh = info["best_rec"]["bbox"]
        tile_data = load_single_tile_data(input_path, tile_path)
        if tile_data is None:
            continue
        best_crop = tile_data[by : by + bh, bx : bx + bw, :, best_z]
        worst_crop = tile_data[by : by + bh, bx : bx + bw, :, worst_z]
        ylabel_str = f"{info['label']}\n(#{ann_idx})"
        axes[i, 0].imshow(best_crop)
        axes[i, 0].set_title(f"Best Z={best_z}  ({metric_label}={info['best_rec'][metric]:.0f})", fontsize=9)
        axes[i, 0].set_ylabel(ylabel_str, fontsize=8, rotation=0, labelpad=45, va="center")
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        axes[i, 1].imshow(worst_crop)
        axes[i, 1].set_title(f"Worst Z={worst_z}  ({metric_label}={info['worst_rec'][metric]:.0f})", fontsize=9)
        axes[i, 1].set_xticks([])
        axes[i, 1].set_yticks([])
    axes[0, 0].text(0.5, 1.35, "Sharpest Focal Plane", transform=axes[0, 0].transAxes, ha="center", fontsize=11, fontweight="bold", color=COLOR_VOL)
    axes[0, 1].text(0.5, 1.35, "Worst Focal Plane", transform=axes[0, 1].transAxes, ha="center", fontsize=11, fontweight="bold", color="#B00020")
    fig.suptitle(f"Best vs Worst Focal Plane - {metric_label}\n({mode} selection, {n} examples)", fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    prefix = "06" if mode == "spread" else "07"
    path = output_dir / f"{prefix}_{mode}_examples_{metric}.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def plot_tile_metric_comparison(ds_records: list[dict[str, Any]], input_path: Path, output_dir: Path, n_examples: int = 4, mode: str = "random") -> None:
    import random

    by_tile: dict[str, list[dict]] = defaultdict(list)
    for rec in ds_records:
        by_tile[rec["tile_path"]].append(rec)
    tile_paths = list(by_tile.keys())
    if mode == "random":
        random.seed(449)
        selected_paths = random.sample(tile_paths, min(n_examples, len(tile_paths)))
    else:
        selected_paths = tile_paths[:n_examples]
    metrics = [("vol", "VoL", COLOR_VOL), ("tenengrad", "Tenengrad", COLOR_TEN), ("pvol", "pVoL", COLOR_PVOL), ("ptenengrad", "pTenengrad", COLOR_PTEN)]
    n = len(selected_paths)
    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    if n == 1:
        axes = np.atleast_2d(axes)
    for i, tile_path in enumerate(selected_paths):
        recs = by_tile[tile_path]
        tile_data, focus_stacked = load_single_tile_data_with_stack(input_path, tile_path)
        if tile_data is None:
            continue
        for j, (metric, label, color) in enumerate(metrics):
            best_z = max(recs, key=lambda r: r[metric])["z_index"]
            score = max(recs, key=lambda r: r[metric])[metric]
            axes[i, j].imshow(tile_data[:, :, :, best_z])
            axes[i, j].set_title(f"{label}\nZ={best_z}  ({score:.0f})", fontsize=9, color=color)
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
        axes[i, 4].imshow(focus_stacked)
        axes[i, 4].set_title("Focus Stack", fontsize=9, color="black")
        axes[i, 4].set_xticks([])
        axes[i, 4].set_yticks([])
        axes[i, 0].set_ylabel(os.path.basename(tile_path), fontsize=7, rotation=0, labelpad=60, va="center")
    fig.suptitle("Best Focal Plane per Metric vs Focus Stack - Full Tile", fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    path = output_dir / "08_tile_metric_comparison.png"
    fig.savefig(path)
    plt.close(fig)
    print("Saved", path)


def export_csv(roi_records: list[dict[str, Any]], ds_records: list[dict[str, Any]], output_dir: Path) -> None:
    roi_csv = output_dir / "focus_scores_per_roi.csv"
    with open(roi_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["tile_path", "ann_idx", "label", "z_index", "vol", "tenengrad", "pvol", "ptenengrad", "bbox"])
        writer.writeheader()
        writer.writerows(roi_records)
    print("Saved", roi_csv)
    ds_csv = output_dir / "focus_scores_dataset_wide.csv"
    with open(ds_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["tile_path", "z_index", "vol", "tenengrad", "pvol", "ptenengrad"])
        writer.writeheader()
        writer.writerows(ds_records)
    print("Saved", ds_csv)

if False:
    WORKERS = 0
    FILTER_CLIPPED = False

    h5_paths = list_h5_paths(str(H5_INPUT))
    if not h5_paths:
        print("No H5 files found at", H5_INPUT)
        raise SystemExit(1)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    if WORKERS > 1:
        n_workers = min(WORKERS, os.cpu_count() or 1)
        print(f"Computing focus scores in parallel ({n_workers} workers)...")
        roi_records, ds_records, n_z = compute_focus_scores_parallel(
            H5_INPUT, filter_clipped=FILTER_CLIPPED, n_workers=n_workers
        )
    else:
        tile_stream = iter_tiles_from_h5(H5_INPUT)
        roi_records, ds_records, n_z = compute_focus_scores(tile_stream, filter_clipped=FILTER_CLIPPED)

    print(f"Per-ROI records: {len(roi_records)}, dataset-wide: {len(ds_records)}, n_z: {n_z}")
    export_csv(roi_records, ds_records, OUTPUT_DIR)

    if roi_records:
        plot_best_z_histogram(roi_records, n_z, OUTPUT_DIR)
        plot_mean_focus_per_z_roi(roi_records, n_z, OUTPUT_DIR)
        plot_metric_agreement_scatter(roi_records, OUTPUT_DIR)
        plot_heatmap(roi_records, n_z, OUTPUT_DIR)
        for metric in ("vol", "tenengrad", "pvol", "ptenengrad"):
            plot_best_worst_examples(roi_records, H5_INPUT, OUTPUT_DIR, n_examples=8, metric=metric, mode="spread")
            plot_best_worst_examples(roi_records, H5_INPUT, OUTPUT_DIR, n_examples=8, metric=metric, mode="random")
    else:
        print("No annotated ROIs; skipping per-ROI plots.")

    if ds_records:
        plot_dataset_wide_focus(ds_records, n_z, OUTPUT_DIR)
        plot_tile_metric_comparison(ds_records, H5_INPUT, OUTPUT_DIR, n_examples=10, mode="random")
    else:
        print("No tiles; skipping dataset-wide plot.")

    print("Done. Output in", OUTPUT_DIR)